In [32]:
# Cell 1: imports, seed, basic config (JAX + Optax, 不用 Flax)

import os, time, random, math
import numpy as np
import jax
import jax.numpy as jnp
import optax
import pandas as pd

# reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    # JAX 的 PRNG 每次在代码里单独传，这里只管 np/random
    # 如果你想，也可以在这里保存一个全局 key

set_seed(42)

FAST_DEV = True

if FAST_DEV:
    DATA_FRACTION = 0.05
    MAX_ITERS     = 1500
    BATCH_SIZE    = 32
    CONTEXT_LEN   = 64
else:
    DATA_FRACTION = 1.0
    MAX_ITERS     = 100_000
    BATCH_SIZE    = 64
    CONTEXT_LEN   = 128

print("FAST_DEV:", FAST_DEV)


FAST_DEV: True


In [33]:
# Cell 2: load text8 subset (train/test)

train_path = "./data/text8_train.txt"
test_path  = "./data/text8_test.txt"

with open(train_path, "r") as f:
    train_text = f.read()
with open(test_path, "r") as f:
    test_text = f.read()

print(f"Original lengths: train={len(train_text):_}, test={len(test_text):_}")

if DATA_FRACTION < 1.0:
    train_text = train_text[: int(len(train_text) * DATA_FRACTION)]
    test_text  = test_text[: int(len(test_text) * DATA_FRACTION)]
    print(f"Using subset: train={len(train_text):_}, test={len(test_text):_}")

print("Sample training text preview:", repr(train_text[:200]))


Original lengths: train=90_000_000, test=5_000_000
Using subset: train=4_500_000, test=250_000
Sample training text preview: ' anarchism originated as a term of abuse first used against early working class radicals including the diggers of the english revolution and the sans culottes of the french revolution whilst the term '


In [34]:
# Cell 3: tokenizer, encode/decode, split

char_set = list("abcdefghijklmnopqrstuvwxyz ")
char_to_int = {ch: i for i, ch in enumerate(char_set)}
int_to_char = {i: ch for ch, i in char_to_int.items()}
vocab_size = len(char_set)
print("vocab_size =", vocab_size)

def encode(s: str) -> np.ndarray:
    return np.array([char_to_int[c] for c in s], dtype=np.int32)

def decode(ids) -> str:
    return "".join(int_to_char[int(i)] for i in ids)

train_ids = encode(train_text)
test_ids  = encode(test_text)

val_ratio = 0.1
split_idx = int(len(train_ids) * (1 - val_ratio))
train_data = train_ids[:split_idx]
val_data   = train_ids[split_idx:]

print(f"train_data={len(train_data):_}, val_data={len(val_data):_}, test_data={len(test_ids):_}")


vocab_size = 27
train_data=4_050_000, val_data=450_000, test_data=250_000


In [35]:
# Cell 4: get_batch helper (JAX arrays)

def get_batch(split: str, batch_size: int, block_size: int):
    data_ = train_data if split == "train" else val_data
    ix = np.random.randint(0, len(data_) - block_size - 1, size=batch_size)
    x = np.stack([data_[i:i+block_size] for i in ix])
    y = np.stack([data_[i+1:i+block_size+1] for i in ix])
    return jnp.array(x, dtype=jnp.int32), jnp.array(y, dtype=jnp.int32)


In [36]:
# Cell 5: 手写 LSTM 参数初始化

def init_lstm_params(rng, vocab_size: int, d_model: int, n_layers: int):
    params = {}
    rng_embed, rng_layers, rng_out = jax.random.split(rng, 3)

    # Embedding
    params["embed"] = jax.random.normal(rng_embed, (vocab_size, d_model)) * 0.01

    # LSTM layers
    layers = []
    in_dim = d_model
    rng_l = rng_layers
    for _ in range(n_layers):
        rng_l, k = jax.random.split(rng_l)
        # W: (in_dim + d_model, 4 * d_model), b: (4 * d_model,)
        W = jax.random.normal(k, (in_dim + d_model, 4 * d_model)) / jnp.sqrt(in_dim + d_model)
        b = jnp.zeros((4 * d_model,), dtype=jnp.float32)
        layers.append({"W": W, "b": b})
        in_dim = d_model  # 每层输出都是 d_model
    params["layers"] = layers

    # Output projection
    params["out_W"] = jax.random.normal(rng_out, (d_model, vocab_size)) / jnp.sqrt(d_model)
    params["out_b"] = jnp.zeros((vocab_size,), dtype=jnp.float32)

    return params


In [37]:
# Cell 6: LSTM forward + logits

def lstm_layer_forward(layer_params, x):
    """
    x: (B, T, d_model)
    returns: (B, T, d_model)
    """
    W = layer_params["W"]    # (in_dim + d_model, 4*d_model)
    b = layer_params["b"]    # (4*d_model,)
    B, T, D = x.shape
    H = W.shape[1] // 4      # hidden size == d_model

    # 输入时间维度挪到前面，方便 scan: (T, B, D)
    x_time_major = jnp.swapaxes(x, 0, 1)

    h0 = jnp.zeros((B, H), dtype=jnp.float32)
    c0 = jnp.zeros((B, H), dtype=jnp.float32)
    init_carry = (h0, c0)

    def step(carry, x_t):
        h, c = carry
        # x_t: (B, D)
        z = jnp.concatenate([x_t, h], axis=-1)  # (B, D + H)
        gates = jnp.matmul(z, W) + b           # (B, 4H)
        i, f, g, o = jnp.split(gates, 4, axis=-1)

        i = jax.nn.sigmoid(i)
        f = jax.nn.sigmoid(f)
        o = jax.nn.sigmoid(o)
        g = jnp.tanh(g)

        c_new = f * c + i * g
        h_new = o * jnp.tanh(c_new)
        return (h_new, c_new), h_new

    (_, _), h_seq = jax.lax.scan(step, init_carry, x_time_major)  # h_seq: (T, B, H)
    h_seq = jnp.swapaxes(h_seq, 0, 1)  # (B, T, H)
    return h_seq

def apply_lstm(params, x_int):
    """
    x_int: (B, T) int32
    returns logits: (B, T, vocab_size)
    """
    # Embedding lookup
    x = params["embed"][x_int]  # (B, T, d_model)

    # LSTM layers
    for layer in params["layers"]:
        x = lstm_layer_forward(layer, x)

    # Output projection
    logits = jnp.einsum("btd,df->btf", x, params["out_W"]) + params["out_b"]
    return logits


In [38]:
# Cell 7: loss_and_metrics + train_step

def loss_and_metrics(params, x, y):
    logits = apply_lstm(params, x)
    vocab = logits.shape[-1]
    flat_logits = logits.reshape(-1, vocab)
    flat_targets = y.reshape(-1)

    per_pos = optax.softmax_cross_entropy_with_integer_labels(flat_logits, flat_targets)
    loss = per_pos.mean()

    preds = jnp.argmax(logits, axis=-1)  # (B, T)
    is_match = (preds == y)
    acc_all = is_match.astype(jnp.float32).mean()
    acc_last = is_match[:, -1].astype(jnp.float32).mean()

    return loss, {"loss": loss, "acc": acc_all, "acc_last": acc_last}

def make_train_step(tx):
    @jax.jit
    def train_step(params, opt_state, x, y):
        (loss, metrics), grads = jax.value_and_grad(loss_and_metrics, has_aux=True)(
            params, x, y
        )
        updates, opt_state = tx.update(grads, opt_state, params)
        new_params = optax.apply_updates(params, updates)
        return new_params, opt_state, metrics
    return train_step


In [39]:
# Cell 8: train_and_eval for a given config

def train_and_eval(cfg, steps=800, seed=42):
    set_seed(seed)
    rng = jax.random.PRNGKey(seed)

    params = init_lstm_params(
        rng,
        vocab_size=vocab_size,
        d_model=cfg["d_model"],
        n_layers=cfg["n_layers"],
    )

    tx = optax.adam(learning_rate=cfg["lr"])
    opt_state = tx.init(params)
    train_step = make_train_step(tx)

    t0 = time.time()
    for it in range(steps):
        xb, yb = get_batch("train", batch_size=cfg["batch_size"], block_size=cfg["context_len"])
        params, opt_state, metrics = train_step(params, opt_state, xb, yb)

    elapsed = time.time() - t0

    # 验证集 acc_last
    xb_val, yb_val = get_batch("val", batch_size=256, block_size=cfg["context_len"])
    _, val_metrics = loss_and_metrics(params, xb_val, yb_val)

    return {
        "val_acc_last": float(val_metrics["acc_last"]),
        "time_sec": elapsed,
    }


In [40]:
# Cell 9: define search grid

SEARCH = {
    "lr":        [1e-3, 5e-4, 1e-4],
    "d_model":   [128, 256],
    "n_layers":  [1, 2],
    "context_len": [32, 64],
}

DEFAULTS = {
    "batch_size": BATCH_SIZE,
}

grid = []
for lr in SEARCH["lr"]:
    for d in SEARCH["d_model"]:
        for nl in SEARCH["n_layers"]:
            for L in SEARCH["context_len"]:
                cfg = DEFAULTS.copy()
                cfg.update({
                    "lr": lr,
                    "d_model": d,
                    "n_layers": nl,
                    "context_len": L,
                })
                grid.append(cfg)

print(f"Total {len(grid)} configs")
for i, cfg in enumerate(grid[:5], 1):
    print(f"Example cfg {i}: {cfg}")


Total 24 configs
Example cfg 1: {'batch_size': 32, 'lr': 0.001, 'd_model': 128, 'n_layers': 1, 'context_len': 32}
Example cfg 2: {'batch_size': 32, 'lr': 0.001, 'd_model': 128, 'n_layers': 1, 'context_len': 64}
Example cfg 3: {'batch_size': 32, 'lr': 0.001, 'd_model': 128, 'n_layers': 2, 'context_len': 32}
Example cfg 4: {'batch_size': 32, 'lr': 0.001, 'd_model': 128, 'n_layers': 2, 'context_len': 64}
Example cfg 5: {'batch_size': 32, 'lr': 0.001, 'd_model': 256, 'n_layers': 1, 'context_len': 32}


In [41]:
# Cell 10: run small experiment grid

results = []
for i, cfg in enumerate(grid, 1):
    print(f"[{i}/{len(grid)}] Testing {cfg}")
    out = train_and_eval(cfg, steps=600, seed=42)
    row = {**cfg, **out}
    print(f"  -> acc_last={row['val_acc_last']*100:.2f}%, time={row['time_sec']:.1f}s")
    results.append(row)

df = (
    pd.DataFrame(results)
      .sort_values("val_acc_last", ascending=False)
      .reset_index(drop=True)
)

print("\n=== LSTM smallExperiment results (top 10) ===")
display(df.head(10))

csv_path = "lstm_smallExperiment_results.csv"
df.to_csv(csv_path, index=False)
print("Saved results to:", csv_path)


[1/24] Testing {'batch_size': 32, 'lr': 0.001, 'd_model': 128, 'n_layers': 1, 'context_len': 32}
  -> acc_last=33.98%, time=8.5s
[2/24] Testing {'batch_size': 32, 'lr': 0.001, 'd_model': 128, 'n_layers': 1, 'context_len': 64}
  -> acc_last=38.67%, time=15.9s
[3/24] Testing {'batch_size': 32, 'lr': 0.001, 'd_model': 128, 'n_layers': 2, 'context_len': 32}
  -> acc_last=35.94%, time=15.8s
[4/24] Testing {'batch_size': 32, 'lr': 0.001, 'd_model': 128, 'n_layers': 2, 'context_len': 64}
  -> acc_last=35.55%, time=31.2s
[5/24] Testing {'batch_size': 32, 'lr': 0.001, 'd_model': 256, 'n_layers': 1, 'context_len': 32}
  -> acc_last=38.28%, time=20.5s
[6/24] Testing {'batch_size': 32, 'lr': 0.001, 'd_model': 256, 'n_layers': 1, 'context_len': 64}
  -> acc_last=43.36%, time=38.2s
[7/24] Testing {'batch_size': 32, 'lr': 0.001, 'd_model': 256, 'n_layers': 2, 'context_len': 32}
  -> acc_last=39.06%, time=36.5s
[8/24] Testing {'batch_size': 32, 'lr': 0.001, 'd_model': 256, 'n_layers': 2, 'context_len'

,batch_size,lr,d_model,n_layers,context_len,val_acc_last,time_sec
0,32,0.0010,256,1,64,0.433594,38.239949
1,32,0.0010,256,2,64,0.425781,75.641230
2,32,0.0010,256,2,32,0.390625,36.540448
3,32,0.0010,128,1,64,0.386719,15.904348
4,32,0.0010,256,1,32,0.382812,20.459416
5,32,0.0010,128,2,32,0.359375,15.808145
6,32,0.0010,128,2,64,0.355469,31.152960
7,32,0.0005,256,1,32,0.343750,18.824455
8,32,0.0005,256,1,64,0.339844,33.350177
9,32,0.0010,128,1,32,0.339844,8.503434


Saved results to: lstm_smallExperiment_results.csv


In [45]:
# Cell 11: Round 2 grid search with advanced params

# Step 1: 选取 Round 1 的 top-2 配置作为基础结构
top_k = 2
base_configs = df.head(top_k).to_dict(orient="records")

# Step 2: 设置 Round 2 搜索维度（基于专业建议）
BETA2_CANDIDATES = [0.90, 0.95, 0.999]
DROPOUT_CANDIDATES = [0.0, 0.1]
WARMUP_CANDIDATES = [0, 200]
LR_JITTER = [0.7, 1.0, 1.3]

# Step 3: 构建 Round 2 搜索网格
round2_grid = []

for base in base_configs:
    for lr_scale in LR_JITTER:
        for beta2 in BETA2_CANDIDATES:
            for drop in DROPOUT_CANDIDATES:
                for warm in WARMUP_CANDIDATES:
                    cfg = {
                        "lr":           base["lr"] * lr_scale,
                        "d_model":      int(base["d_model"]),
                        "n_layers":     int(base["n_layers"]),
                        "context_len":  int(base["context_len"]),
                        "batch_size":   int(base["batch_size"]),
                        "beta2":        beta2,
                        "dropout":      drop,
                        "warmup_steps": warm,
                    }
                    round2_grid.append(cfg)

print(f"Round 2 total: {len(round2_grid)} configs")
for i, c in enumerate(round2_grid[:5]):
    print(f"Config {i+1}: {c}")


Round 2 total: 72 configs
Config 1: {'lr': 0.0007, 'd_model': 256, 'n_layers': 1, 'context_len': 64, 'batch_size': 32, 'beta2': 0.9, 'dropout': 0.0, 'warmup_steps': 0}
Config 2: {'lr': 0.0007, 'd_model': 256, 'n_layers': 1, 'context_len': 64, 'batch_size': 32, 'beta2': 0.9, 'dropout': 0.0, 'warmup_steps': 200}
Config 3: {'lr': 0.0007, 'd_model': 256, 'n_layers': 1, 'context_len': 64, 'batch_size': 32, 'beta2': 0.9, 'dropout': 0.1, 'warmup_steps': 0}
Config 4: {'lr': 0.0007, 'd_model': 256, 'n_layers': 1, 'context_len': 64, 'batch_size': 32, 'beta2': 0.9, 'dropout': 0.1, 'warmup_steps': 200}
Config 5: {'lr': 0.0007, 'd_model': 256, 'n_layers': 1, 'context_len': 64, 'batch_size': 32, 'beta2': 0.95, 'dropout': 0.0, 'warmup_steps': 0}


In [46]:
# Cell 12: train_and_eval_lstm_round2

def train_and_eval_lstm_round2(cfg, steps=600, seed=42):
    set_seed(seed)
    rng = jax.random.PRNGKey(seed)

    params = init_lstm_params(
        rng,
        vocab_size=vocab_size,
        d_model=cfg["d_model"],
        n_layers=cfg["n_layers"],
    )

    # Use beta2 from config
    tx = optax.adam(learning_rate=cfg["lr"], b2=cfg["beta2"])
    opt_state = tx.init(params)
    train_step = make_train_step(tx)

    # warmup schedule
    def lr_schedule(step):
        if step < cfg["warmup_steps"]:
            return cfg["lr"] * (step + 1) / max(1, cfg["warmup_steps"])
        return cfg["lr"]

    t0 = time.time()
    for it in range(steps):
        lr_now = lr_schedule(it)

        xb, yb = get_batch("train", batch_size=cfg["batch_size"], block_size=cfg["context_len"])
        # apply dropout manually to input?
        if cfg["dropout"] > 0.0:
            mask = jax.random.bernoulli(jax.random.PRNGKey(seed+it), p=1.0 - cfg["dropout"], shape=xb.shape)
            xb = xb * mask

        params, opt_state, metrics = train_step(params, opt_state, xb, yb)

    elapsed = time.time() - t0

    xb_val, yb_val = get_batch("val", batch_size=256, block_size=cfg["context_len"])
    _, val_metrics = loss_and_metrics(params, xb_val, yb_val)

    return {
        "val_acc_last": float(val_metrics["acc_last"]),
        "time_sec": elapsed,
    }


In [47]:
# Cell 13: run Round 2 grid and save results

results_round2 = []
for i, cfg in enumerate(round2_grid, 1):
    print(f"[Round 2: {i}/{len(round2_grid)}] cfg = {cfg}")
    out = train_and_eval_lstm_round2(cfg, steps=600)
    row = {**cfg, **out}
    print(f"  --> acc_last = {row['val_acc_last']*100:.2f}%, time = {row['time_sec']:.1f}s")
    results_round2.append(row)

df2 = (
    pd.DataFrame(results_round2)
      .sort_values("val_acc_last", ascending=False)
      .reset_index(drop=True)
)

print("\n=== Round 2 results (top 10) ===")
display(df2.head(10))

df2.to_csv("lstm_round2_results.csv", index=False)
print("Saved to lstm_round2_results.csv")


[Round 2: 1/72] cfg = {'lr': 0.0007, 'd_model': 256, 'n_layers': 1, 'context_len': 64, 'batch_size': 32, 'beta2': 0.9, 'dropout': 0.0, 'warmup_steps': 0}
  --> acc_last = 38.67%, time = 31.0s
[Round 2: 2/72] cfg = {'lr': 0.0007, 'd_model': 256, 'n_layers': 1, 'context_len': 64, 'batch_size': 32, 'beta2': 0.9, 'dropout': 0.0, 'warmup_steps': 200}
  --> acc_last = 38.67%, time = 29.9s
[Round 2: 3/72] cfg = {'lr': 0.0007, 'd_model': 256, 'n_layers': 1, 'context_len': 64, 'batch_size': 32, 'beta2': 0.9, 'dropout': 0.1, 'warmup_steps': 0}
  --> acc_last = 36.72%, time = 31.3s
[Round 2: 4/72] cfg = {'lr': 0.0007, 'd_model': 256, 'n_layers': 1, 'context_len': 64, 'batch_size': 32, 'beta2': 0.9, 'dropout': 0.1, 'warmup_steps': 200}
  --> acc_last = 36.72%, time = 31.9s
[Round 2: 5/72] cfg = {'lr': 0.0007, 'd_model': 256, 'n_layers': 1, 'context_len': 64, 'batch_size': 32, 'beta2': 0.95, 'dropout': 0.0, 'warmup_steps': 0}
  --> acc_last = 38.28%, time = 32.0s
[Round 2: 6/72] cfg = {'lr': 0.0007

,lr,d_model,n_layers,context_len,batch_size,beta2,dropout,warmup_steps,val_acc_last,time_sec
0,0.0013,256,2,64,32,0.900,0.0,0,0.488281,65.948833
1,0.0013,256,2,64,32,0.900,0.0,200,0.488281,66.978693
2,0.0013,256,1,64,32,0.999,0.0,200,0.480469,35.323308
3,0.0013,256,1,64,32,0.999,0.0,0,0.480469,34.472273
4,0.0013,256,2,64,32,0.950,0.0,200,0.476562,63.558215
5,0.0013,256,2,64,32,0.950,0.0,0,0.476562,69.438844
6,0.0013,256,1,64,32,0.950,0.0,200,0.460938,33.340458
7,0.0013,256,1,64,32,0.950,0.0,0,0.460938,33.157577
8,0.0013,256,1,64,32,0.900,0.0,200,0.449219,32.881474
9,0.0013,256,1,64,32,0.900,0.0,0,0.449219,32.167804


Saved to lstm_round2_results.csv
